# Project: Fire Mapping

Goal: Build an HTML page with an interactive map where the user can see the recent fires. Provide the user with information of physical size, duration, intensity, etc. of the fires. Use pop-ups and tooltips to make maps interactive and structured for the users. The maps are designed for public users who are interesting in fires.

Tasks:
1. Load the api
2. Extract the data which is used to locate the fire (VIIRS_SNPP_NRT)
3. Explore the data
4. Clean the data
5. Check fires for different properties
6. Visualization of the fires
7. Provide additional information about the fires. 
8. Make the map interactive

---
### Set up the environment
Make sure all libraries are installed in the python envirnonment used to run this notebook. Also create the folder structure that the outputs can be safed in an secure way
  
***Read the README.md file before running this notebook***

In [ ]:
# standard libraries
import io
import json
import os

# third-party libraries
# conda: conda install -c conda-forge geopandas folium plotly shapely python-dotenv
# pip:   pip install geopandas folium plotly shapely python-dotenv
import folium
import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
from dotenv import load_dotenv
from folium.plugins import MarkerCluster
from shapely.geometry import MultiPoint

# creates folder outputs if folder does not exist now
os.makedirs("../outputs", exist_ok=True)

---
### Creating the API and its key
After all libraries are imported, the next step is to configure the API URL and its key.  
Create the key on: https://firms.modaps.eosdis.nasa.gov/api/map_key/  
Save the map key in an .env file with the variable "FIRMS_API_KEY"


In [ ]:
# import the api key from the .env environment
load_dotenv()

#get the api key from the .env file
api_key = os.environ.get("FIRMS_API_KEY")
if not api_key:
    raise ValueError("FIRMS_API_KEY not set. Check your .env file.")

# define source
api_source = "VIIRS_SNPP_NRT" # VIIRS sensor on Suomi-NPP satellite, Near Real-Time data (last 1–5 days)
# define area coordinates
api_area_coordinates = "world"
# day range. Days going back from today. 
api_day_range = 5 # Note: Day range must be an int between 1 and 5
# build api url with api key
api_url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{api_key}/{api_source}/{api_area_coordinates}/{api_day_range}"

After successfully loading the API URL consisting of the source, location,day range and key the data is ready to be requested.

In [ ]:
# load the data form the api
response = requests.get(api_url)

# check if api import was successfull
if response.status_code == 200:
    print("API request successfull")

    # get the data as a csv
    data_csv = response.text
    # create a dataframe
    data_df = pd.read_csv(io.StringIO(data_csv))

    # check if the import contains any data
    if data_df.empty:
        raise ValueError("API returned no data. Check your key or quota.")

else:
    raise ConnectionError(f"API request failed. Status code: {response.status_code}")

After successfully loading the data from the API, in the next step the data gets referenced as a GeoDataFrame, named appropriately, get cleaned and adjusted with a datetime format.  
Additionally, spatial buffer are created around each fire detection. The goal of this buffering is to combine fires occuring on the same day within close proximity. With this adjustment,nearby fires will be referrd as 1 fire and not as several. This is because it's likely that there are not two small fires but rather one big fire in this areas. This makes the maps that are build later less crowded and therefore more easy to read and understand. There might be a loss in the visibility of fires in large areas, but we will take care of this in a later step

In [ ]:
# converting the data data frame into a geo-data frame
fire_gdf_crs4326 = gpd.GeoDataFrame(data_df, geometry=gpd.points_from_xy(data_df["longitude"], data_df["latitude"]), crs=4326)

# convert the date column into a date datetime type
fire_gdf_crs4326["acq_date"] = pd.to_datetime(fire_gdf_crs4326["acq_date"], format="%Y-%m-%d")

# add a column with the date
fire_gdf_crs4326["acq_date_day"] = fire_gdf_crs4326["acq_date"].dt.date
# Reproject to EPSG:3857 for metre-based buffering
fire_gdf_crs3857 = fire_gdf_crs4326.to_crs(epsg=3857)

# Buffer each fire point by 1501m
fire_gdf_crs3857["geometry_buffer"] = fire_gdf_crs3857.geometry.buffer(1501)

# Spatial join: for each buffer, find all fire points that fall within it
joined = gpd.sjoin(
    fire_gdf_crs3857[["acq_date", "geometry"]].reset_index(names="point_idx"),
    fire_gdf_crs3857[["acq_date", "geometry_buffer"]].reset_index(names="buffer_idx").set_geometry("geometry_buffer"),
    how="inner",           # only keep actual matches
    predicate="within"
)

# Keep only same-day matches
joined = joined[joined["acq_date_left"] == joined["acq_date_right"]]

# Build a mapping: each point_idx → the lowest buffer_idx it belongs to (its cluster ID)
# This ensures nearby points that share buffers end up in the same cluster
cluster_map = joined.groupby("point_idx")["buffer_idx"].min()

# Assign cluster ID to each original point
fire_gdf_crs3857["cluster_id"] = fire_gdf_crs3857.index.map(cluster_map)

# Points that matched nothing keep their own index as cluster ID
fire_gdf_crs3857["cluster_id"] = fire_gdf_crs3857["cluster_id"].fillna(
    pd.Series(fire_gdf_crs3857.index, index=fire_gdf_crs3857.index)
).astype(int)

# Aggregate: one row per cluster, centroid as geometry
fire_clusters = (
    fire_gdf_crs3857.groupby("cluster_id").agg(
        acq_date=("acq_date", "first"),
        count=("acq_date", "count"),
        geometry=("geometry", lambda geoms: MultiPoint(list(geoms)).centroid)
    )
    .reset_index(drop=True)
)

# Rebuild GeoDataFrame and reproject back to WGS84
fire_clustered = gpd.GeoDataFrame(fire_clusters, geometry="geometry", crs=3857).to_crs(epsg=4326)

print(f"Raw fire detections : {len(fire_gdf_crs4326):,}")
print(f"Clustered fires     : {len(fire_clustered):,}")
print(f"Reduction           : {1 - len(fire_clustered) / len(fire_gdf_crs4326):.1%}")

In the next section we check the properties of the loaded fires from the API and form the clusterd fires. In this step, you can verify if the data import and cleaning happend as corectly as supposed. 

In [ ]:
# verify transformation to GeoDataFrame
display(fire_gdf_crs4326.head(5))
display(fire_clustered.head(5))
display(fire_clustered.info())
print(f"Raw fire detections: {len(fire_gdf_crs4326)} | Clustered fires: {len(fire_clustered)}")

---
### Building a map showing the recent fire occurrence on the world
All the data required to build our first map of fire locations is now available in the notebook.
In the next section an interactive map is produced. The map will show where the fire was detected and how many fire detections were clusterd in this location. The number of Detections will give additional information about the spread of the fire. Additionally, a second layer to just show the bigger fires is created, where the Clusters are containing 5 or more detected fires. 

In [ ]:
# initalize Folium back ground map
fire_map = folium.Map(
    location=[0, 0], # start zoom at latitude and longitude 0
    zoom_start=2, # shows the whole word at the start
    tiles="CartoDB DarkMatter", # Dark basmap
    control_scale=True # Add scalebar
)

# Layer that shows recent fires with low clusters
cluster_low = MarkerCluster(name="Low Satellite Fire Detection (<5)", show=True).add_to(fire_map)

# Layer shows just fires where the cluster detect 5 or more fire
cluster_5 = MarkerCluster(name="High Satellite Fire Detection (≥5)", show=False).add_to(fire_map)

# build markers for the fires
for idx, row in fire_clustered.iterrows():
    lat = row.geometry.y # extract latitude out of the geometry column 
    lon = row.geometry.x # extract longitude out of the geometry column
    count = row["count"] # counting how many fires are aggregated

    # Formating the Tooltip. Date of the fire registation
    fire_data = row["acq_date"].date()
    tooltip = f"Date: {fire_data} | Detections: {count}"

    # Define Marker color depending the count of fires
    if count == 1:
        color =  "orange" # orange for location where 1 fire is detected
    elif count <= 5:
        color = "red" # red for location where 2 to 5 fires are detected
    else:
        color = "darkred" # darkred for location where more than 5 fires are detected

    # create the the fire Icon
    fire_icon = folium.Icon(color=color, icon="fire", prefix="fa")

    # create Marker of the fire locations
    folium.Marker(
        location=[lat, lon],
        tooltip=tooltip,
        icon=fire_icon
    ).add_to(cluster_low)

    # Add to "High detection" layer only if count >= 5
    if count >= 5:
        folium.Marker(
            location=[lat, lon],
            tooltip=tooltip,
            icon=fire_icon
        ).add_to(cluster_5)

folium.LayerControl().add_to(fire_map)
fire_map.save("../outputs/fire_map.html")
print("Map saved to ../outputs/fire_map.html")

---
### Analysis of the Distribution of Fires
In addition to the map I also decided to give some Insights about in which region most open fires are recorded. To begin this analysis in a first step, data which provide data about the shape of each country must be loaded. Fortunately, Natural Earth (naturalearthdata.com) provides data about each country, the borders and its location.  
Since the data form naturalearthdata.com is a huge dataset, the data is filtered just for data thats intersting for the project. This are the Name of the Countries, the Continent of each country and its geometry. Next, the geographical area of each country is calculated from its geometry. This area metric is critical for computing fire density per country later on. Finally, this dataset is inspected to ensure it loaded accurately.  
Also this data gets inspected to check if its loaded correctly.

In [ ]:
# load Geo Data frame of the world countries
world_import = gpd.read_file("https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_0_countries.zip").to_crs(epsg=4326) # load Countries from online source.

# inspect world gdf
print(world_import.columns.tolist())
print(world_import[["NAME", "CONTINENT", "geometry"]].head(5))

# cleaning the world 
world = world_import[["NAME", "CONTINENT", "geometry"]].copy() # just keep useful attributes ot of the world gdf.
world["AREA_KM2"] = world.to_crs(epsg=8857).geometry.area / 1e6 # add and calculate new column with the area. epsg=8857 for the equal earth projection. dividing with 1e6 to get the area from m2 to km2
world.to_crs(epsg=3857)
display(world)

After the succesful import of the countries poroperties, to each fire a the country of its occurence gets assigned. In this step some fires might get lost. This is due to some fires occure/are recorded in on non territorials areas such like on the ocean or in the Arctic. 
  
To get the numbers of fires per country the dataset with the fires and its corresponding name must be grouped by the country. To capture all countries (and some accepted territories), the countries with zero fire detections must be added with zero recordings. The reason this is that in a later step on the map this countries without any fire recordings also can be implemented.

In [ ]:
# calculate fires in the diffrent countries
# here the fire cluster is used, so nearby fires are counted as one fire and not as several

fire_countries = gpd.sjoin(fire_clustered, world, 
                                how="left", predicate="within") # assign to each fire in which country it is located

# Check for fires not located inside a country for example this are burning ships, fires in artica or antarctica
unmatched = fire_countries[fire_countries["NAME"].isna()]
print(f"Unmatched fires: {len(unmatched)}")
display(fire_countries.head(5))

# count the fires for each country
fire_per_country = world[["NAME", "CONTINENT", "geometry", "AREA_KM2"]].copy() # copy world to build fire per country
# merge fire counts in — countries with no fires get NaN
fire_per_country = fire_per_country.merge(
    fire_countries.groupby("NAME").size().reset_index(name="fire_count"),
    on="NAME",
    how="left"  # keep all countries including countries with 0 fires
)

fire_per_country["fire_count"] = fire_per_country["fire_count"].fillna(0).astype(int) # fill the NaN values of the countries without fires with 0

fire_per_country = gpd.GeoDataFrame(fire_per_country, geometry="geometry", crs=4326) 

display(fire_per_country)

---
### Mapping Fire per Country
In this section the map of the fires per country gets build. Here the area of the countries is not considered. This means that larger countries tend to have more fires, because there is more potential areas for fire. In the section after this one the countries size will be considered.  


In [ ]:
# Shared colorscale used across all choropleth maps: grey for zero, warm gradient for fire counts
fire_colorscale = [
    [0,      "lightgrey"],
    [0.0001, "#fff7bc"],
    [0.3,    "#fd8d3c"],
    [0.6,    "#e31a1c"],
    [1,      "#67000d"]
]

In [ ]:
# make a map displaying the numbers of fire per each country
geojson = json.loads(fire_per_country.to_json()) # provides the shape for the countries.

fig = go.Figure(go.Choropleth(
    geojson=geojson,
    locations=fire_per_country["NAME"],
    featureidkey="properties.NAME",
    z=fire_per_country["fire_count"],
    colorscale=fire_colorscale,
    colorbar=dict(
        title=dict(text="Recorded Fires", side="right", font=dict(size=16, weight="bold")),
    ),
    customdata=fire_per_country[["NAME", "fire_count"]].values,
    hovertemplate="<b>%{customdata[0]}</b><br>Recorded Fires: %{customdata[1]}<extra></extra>",
    marker_line_color="rgba(0,0,0,0.3)",
    marker_line_width=0.5
))

fig.update_layout(
    title=dict(text="Fires per Country", x=0.5, xanchor="center",yanchor="top", font=dict(size=32, weight="bold")),
    margin=dict(l=0, r=0, t=50, b=0),
    geo=dict(
        showframe=True,
        framecolor="grey",
        showland=True,
        landcolor="lightblue", # same color as ocean.
        showcoastlines=False,
        showocean=True,
        oceancolor="lightblue",
        showlakes=True,
        lakecolor="lightblue",
        lataxis=dict(showgrid=True, gridcolor="grey", dtick=30),
        lonaxis=dict(showgrid=True, gridcolor="grey", dtick=60),
        projection_type="natural earth"
    )
)

fig.write_html("../outputs/fire_per_country.html")
print("Map saved to ../outputs/fire_per_country.html")

---
### Mapping the Fire considering Country Size
As already mentioned, just the fire per country are not really informative. Large countries tend to recorded many fires where in smaller countries less fire tend to be recorded. Thats why in this section the density of the fire per country gets introduced. This give some good insight in which country the amount of covering a lot of area. However, since we still work with the clustered fire, this map will not provide any fix information of the area of fire detection. One big fire is counted as with the same value as a small fire.  
  
Because the distrubution of in the fire density is right-skewed mapping with linear scale does show many countries with a low density and just highlights the few country with a high density. Therefore, also the density with a logarithmic scale is calculated. Still the distribution is right-skewed but much better distributed than with the linear scale

In [ ]:
# make a map with density of fires per country
fire_density_per_country = fire_per_country.copy() 
fire_density_per_country["density_100km2"] = fire_density_per_country["fire_count"] / (fire_density_per_country["AREA_KM2"]) *10000 # add column: fire density per 100 km2

# add log column, log(0) is undefined so use log(x+1)
fire_density_per_country["density_log"] = np.log1p(fire_density_per_country["density_100km2"])

# inspect the distributen of the density
fire_density_per_country.describe()

In the next section the Density map showing fires per 100km2 is build. This map will show the fires density in a linear scale. As mentioned before, a lot of countries have a low density and therefore will be diffucult to distinguish.

In [ ]:
# render map linear scale
geojson_density = json.loads(fire_density_per_country.to_json())


fig = go.Figure(go.Choropleth(
    geojson=geojson_density,
    locations=fire_density_per_country["NAME"],
    featureidkey="properties.NAME",
    z=fire_density_per_country["density_100km2"],
    colorscale=fire_colorscale,
    colorbar=dict(
        title=dict(text="Fires per 100km²", side="right", font=dict(size=16, weight="bold"))
    ),
    customdata=fire_density_per_country[["NAME", "fire_count", "density_100km2"]].values,
    hovertemplate="<b>%{customdata[0]}</b><br>Fires recorded: %{customdata[1]}<br>Fires per 100km²: %{customdata[2]:.2f}<extra></extra>",
    marker_line_color="rgba(0,0,0,0.3)",
    marker_line_width=0.5
))

fig.update_layout(
    title=dict(text="Fire Density per 100km² by Country", x=0.5, xanchor="center",yanchor="top", font=dict(size=32, weight="bold")),
    margin=dict(l=0, r=0, t=50, b=0),
    geo=dict(
        showframe=True,
        framecolor="grey",
        showland=True,
        landcolor="lightblue", # same color as ocean
        showcoastlines=False,
        showocean=True,
        oceancolor="lightblue",
        showlakes=True,
        lakecolor="lightblue",
        lataxis=dict(showgrid=True, gridcolor="grey", dtick=30),
        lonaxis=dict(showgrid=True, gridcolor="grey", dtick=60),
        projection_type="natural earth"
    )
)

fig.write_html("../outputs/fire_density_per_country_linear.html")
print("Map saved to ../outputs/fire_density_per_country_linear.html")

Since the density fire map with the linear scale is not very informative. The next section will be about the logarithmic scale. One disadvantage of this map is, that just the colorscale is not even over its length. In the high density part just a small colorchange means a large increase in fire density, while in the lower scale a small color change is not such a big increase in fire density.

In [ ]:
# render map logarithmic scale
geojson_density = json.loads(fire_density_per_country.to_json())

# Evenly spaced real values from 0 to max, converted to log scale for positioning
max_log = fire_density_per_country["density_log"].max()
tick_log  = np.linspace(0, max_log, 6)
# Convert back to real values for the labels
tick_real_raw = np.expm1(tick_log)
tick_real = np.where(tick_real_raw < 10, # small values (<10) are rounded to int number and large vaues are rounded to nearest 10
                     tick_real_raw.round(0),    # round to int number
                     tick_real_raw.round(-1)    # nearest 10 for large values
            )

fig = go.Figure(go.Choropleth(
    geojson=geojson_density,
    locations=fire_density_per_country["NAME"],
    featureidkey="properties.NAME",
    z=fire_density_per_country["density_log"],
    colorscale=fire_colorscale,
    colorbar=dict(
        title=dict(text="Fires per 100km²", side="right", font=(dict(size=16, weight="bold"))),
        tickvals=tick_log, # tick positions are in log scale
        ticktext=[str(v) for v in tick_real]), # show the real numbers of fires at the legend bar
    customdata=fire_density_per_country[["NAME", "fire_count", "density_100km2"]].values,
    hovertemplate="<b>%{customdata[0]}</b><br>Fires recorded: %{customdata[1]}<br>Fires per 100km²: %{customdata[2]:.2f}<extra></extra>",
    marker_line_color="rgba(0, 0, 0, 0.3)",
    marker_line_width=0.5
))

fig.update_layout(
    title=dict(text="Fire Density per 100km² by Country (Log Scale)", x=0.5, xanchor="center", yanchor="top", font=dict(size=32, weight="bold")),
    margin=dict(l=0, r=0, t=50, b=0),
    geo=dict(
        showframe=True,
        framecolor="grey",
        showland=True,
        landcolor="lightblue", # same color as ocean
        showcoastlines=False,
        showocean=True,
        oceancolor="lightblue",
        showlakes=True,
        lakecolor="lightblue",
        lataxis=dict(showgrid=True, gridcolor="grey", dtick=30),
        lonaxis=dict(showgrid=True, gridcolor="grey", dtick=60),
        projection_type="natural earth"
    )
)

fig.write_html("../outputs/fire_density_per_country_logarithmic.html")
print("Map saved to ../outputs/fire_density_per_country_logarithmic.html")


---
### Fire Heatmap
Till now we just looked at the different fires and the countries within the fire was recorded. As we all know a fire migh not just stop at a countries border (if the border isn't a big river, lake or high-mountain ridge). For a whole overview the see which regions are affected by fire, the heat map, which is build in the next section is very useful. The heatmap will highlight region where many fires were recorded. For this case the cleaned loaded api data is used and not the clustered one. This helps to really highlight the fires even zoomed in on a smaller scale. 

In [ ]:
# make a heatmap of the fires
fig = px.density_map(
    fire_gdf_crs4326, # using the non-custerd data for a denser and more acurate heatmap.
    lat=fire_gdf_crs4326.geometry.y,
    lon=fire_gdf_crs4326.geometry.x,
    radius=5,
    zoom=1,
    map_style="carto-darkmatter",
    title="Fire Heatmap",
    color_continuous_scale="Hot",
    hover_data={"acq_date_day": True}
)

fig.update_traces(hovertemplate="Date: %{customdata[0]}<extra></extra>")

fig.update_layout(
    title=dict(text="Global Fire Heatmap", x=0.5, xanchor="center", yanchor="top", font=dict(size=32, weight="bold")),
    margin=dict(l=0, r=0, t=70, b=0),
    coloraxis_showscale=False,
)

fig.write_html("../outputs/fire_heatmap.html")

---
### From visual to numbers
The maps before just provide some visual information. It is really difficult to get the real number out of the maps. Therefore, in this a dataset is created taking the numbers for each country. This lists are sorted by the numbers of fires/density per 100km2.

In [ ]:
# Sort and export the fire per country
fire_per_country.sort_values("fire_count", ascending=False)[["NAME", "fire_count"]].to_csv("../outputs/fire_per_country.csv", index=False)
print("Exported fire_per_country.csv")

# sort and export the country with most fire per density
fire_density_per_country.sort_values("density_100km2", ascending=False)[["NAME", "density_100km2", "AREA_KM2"]].to_csv("../outputs/fire_density_per_country.csv", index=False)
print("Exported fire_density_per_country.csv")


---
### The end
This is the end of the notbook. You successfully created maps about the recent fires in the world. All maps are saved in the project's outputs folder. Do you see any difference between the absolute numbers of fire per country and the fire density per country?  These maps look different. Normalizing by country size offers a significantly different spatial narrative than using unweighted counts.